# How Many Clusters? — evaluating without labels

We chose `k = 3` for the penguins because we already knew the answer. That is not clustering,
that is checking.

This notebook is about the genuine problem: **how do you choose `k`, and how do you tell
whether a clustering is any good, when there is nothing to compare against?**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (
    adjusted_mutual_info_score,
    adjusted_rand_score,
    silhouette_samples,
    silhouette_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

RSEED = 42

penguins = pd.read_csv("data/penguins.csv").dropna(
    subset=["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
)
measurements = penguins[["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]]
X = StandardScaler().fit_transform(measurements)

## The elbow method, and why it disappoints

Inertia always falls as `k` rises — with one cluster per point it reaches zero. So you cannot
just minimise it.

The elbow method says: plot inertia against `k` and look for the bend where the improvement
stops being worth it.

In [ ]:
ks = range(1, 11)
inertias = [KMeans(n_clusters=k, n_init=10, random_state=RSEED).fit(X).inertia_ for k in ks]

plt.plot(ks, inertias, marker="o", color="#F5821F", lw=2.5)
plt.xlabel("k")
plt.ylabel("inertia")
plt.title("The elbow method — where does it bend?")
plt.grid(alpha=0.3)
plt.show()

for k, inertia in zip(ks, inertias):
    drop = "" if k == 1 else f"   (−{inertias[k - 2] - inertia:.0f})"
    print(f"k={k:<3} inertia {inertia:7.1f}{drop}")

Look at the drops rather than the curve. There is a big fall from 1 to 2, a decent one from
2 to 3, and after that they get steadily smaller with no obvious cliff.

**That is the honest experience of the elbow method.** It sometimes shows a sharp corner; more
often it shows a gentle bend that three people will read three ways. It is a starting point,
not an answer.

## Silhouette score

A better tool. For each point, compare:

- **a** — its mean distance to the other points in *its own* cluster
- **b** — its mean distance to the points in the *nearest other* cluster

$$s = \frac{b - a}{\max(a, b)}$$

Close to **+1** means the point sits comfortably inside its cluster and far from any other.
Around **0** means it is on a boundary. **Negative** means it is closer to a different
cluster than to its own — it is probably in the wrong one.

The **silhouette score** is the average of `s` over every point, and unlike inertia it does
*not* automatically improve with more clusters.

In [ ]:
for k in range(2, 8):
    labels = KMeans(n_clusters=k, n_init=10, random_state=RSEED).fit_predict(X)
    score = silhouette_score(X, labels)
    bar = "█" * int(score * 60)
    print(f"k={k}  silhouette {score:.4f}  {bar}")

Now there is something to choose on. Read the peak.

### The per-point view

The average hides the shape. A **silhouette plot** shows every point's score, grouped by
cluster — one horizontal bar per point, sorted.

In [ ]:
best_k = 2
labels = KMeans(n_clusters=best_k, n_init=10, random_state=RSEED).fit_predict(X)
values = silhouette_samples(X, labels)

fig, ax = plt.subplots(figsize=(8, 4))
y_lower = 0
for cluster in range(best_k):
    cluster_values = np.sort(values[labels == cluster])
    y_upper = y_lower + len(cluster_values)
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_values, alpha=0.75)
    ax.text(-0.05, y_lower + len(cluster_values) / 2, str(cluster))
    y_lower = y_upper

ax.axvline(silhouette_score(X, labels), color="#F5821F", ls="--", label="mean")
ax.set_xlabel("silhouette coefficient")
ax.set_ylabel("points, grouped by cluster")
ax.set_title(f"Silhouette plot, k={best_k}")
ax.legend()
plt.show()

print(f"points with a negative silhouette: {(values < 0).sum()} of {len(values)}")

What to look for: clusters of **similar width** (similar size), all reaching **past the mean
line**, and **few negative** values. A cluster that is entirely below the mean is a cluster
that should probably not exist.

## When you *do* have labels

Sometimes labels exist and you are clustering anyway — to check whether the structure you
believe in is actually in the data, or to evaluate a clustering method before using it
somewhere labels are absent.

**Do not use accuracy.** Cluster numbers are arbitrary: a perfect clustering that happens to
call the Adelies "2" instead of "0" would score terribly. You need a metric that compares
*partitions* rather than labels.

In [ ]:
species = penguins["species"]

for k in [2, 3, 4]:
    labels = KMeans(n_clusters=k, n_init=10, random_state=RSEED).fit_predict(X)
    print(f"k={k}   AMI {adjusted_mutual_info_score(species, labels):.4f}   "
          f"ARI {adjusted_rand_score(species, labels):.4f}   "
          f"silhouette {silhouette_score(X, labels):.4f}")

**Adjusted Mutual Information** and **Adjusted Rand Index** both ask: do these two
partitions group the same points together? Both are adjusted so that random labelling scores
about 0, and a perfect match scores 1. Neither cares what the clusters are *called*.

And notice the tension in that table: the silhouette score prefers one `k`, while agreement
with the true species prefers another.

**That is not a contradiction — the two measure different things.** Silhouette asks "are
these clusters geometrically tidy?"; AMI asks "do they match the thing I already believe in?"
When you have no labels, only the first question is available, and this is a good reminder
that a tidy clustering and a *meaningful* one are not the same.

## Clustering text

One common use of clustering is on documents, where the features have to be built first.
`CountVectorizer` turns text into one column per word, holding the count of that word.

> **A note on the data.** Skolar's version of this used a Wikipedia news dataset that does
> not ship with this repository. The small corpus below is generated so the notebook runs
> offline; the technique is identical on real text.

In [ ]:
documents = [
    "the striker scored a goal in the second half of the match",
    "the goalkeeper saved a penalty to win the match for his team",
    "our team lost the football match after a late goal",
    "the midfielder passed the ball and the striker scored again",
    "the compiler reported a type error in the function signature",
    "this function returns a list of integers from the array",
    "the variable was assigned before the loop executed correctly",
    "a syntax error in the code stopped the compiler from running",
    "the recipe needs butter flour sugar and two large eggs",
    "bake the cake for forty minutes until the sponge is golden",
    "whisk the eggs and sugar before folding in the flour",
    "add butter to the pan and fry the onions until golden",
]

vectoriser = CountVectorizer(stop_words="english")
doc_features = vectoriser.fit_transform(documents)

print(f"{doc_features.shape[0]} documents × {doc_features.shape[1]} distinct words")

doc_labels = KMeans(n_clusters=3, n_init=10, random_state=RSEED).fit_predict(doc_features.toarray())

for cluster in range(3):
    print(f"\ncluster {cluster}:")
    for i, text in enumerate(documents):
        if doc_labels[i] == cluster:
            print(f"   {text[:58]}")

Three topics, largely recovered from nothing but word counts — no labels, no supervision.

**Largely.** Eleven of the twelve documents landed correctly; *"the variable was assigned
before the loop executed correctly"* was filed with the recipes. Look at why: it shares no
word with the other programming documents. They talk about *compiler*, *error*, *function*;
this one talks about *variable*, *assigned*, *loop*. Bag-of-words has no idea those are
related concepts — it only sees whether the same strings appear.

That is the honest limitation of counting words on a tiny corpus. With more documents the
vocabulary overlaps and the problem largely goes away; `TfidfVectorizer`, which the
Dimensionality Reduction lesson uses, also helps by down-weighting words that appear
everywhere.